# Demo 3 — Building an MCP Tool Server

We'll author a tiny MCP server (`mcp_server.py`) exposing two research tools — a (mocked) clinical-trials search and a (mocked) variant annotator — and let a Microsoft Agent Framework agent (backed by Microsoft Foundry) call them over stdio.

**Prereqs**
- `pip install -r requirements.txt`
- A populated `.env`
- `az login`

## 1. Peek at the server

The MCP server is in `mcp_server.py`. It defines two tools using `FastMCP`:
- `clinical_trials_search(condition, phase=None)`
- `variant_annotation(gene, variant)`

In [ ]:
with open("mcp_server.py", "r", encoding="utf-8") as f:
    print(f.read())

## 2. Load environment

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
print("Foundry project:", PROJECT_ENDPOINT)

## 3. Wire the MCP server into a Foundry agent

`MCPStdioTool` launches our server as a subprocess and exposes its tools to the agent.

In [ ]:
from agent_framework import MCPStdioTool
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
    credential=credential,
)

mcp_tools = MCPStdioTool(
    name="pediatric-onc-research-tools",
    command=sys.executable,
    args=["mcp_server.py"],
)

agent = client.as_agent(
    name="OncoResearchMCPAgent",
    instructions=(
        "You are a pediatric oncology research assistant. Use the available MCP tools "
        "to look up open clinical trials and annotate gene variants. Prefer tool calls "
        "over your own recollection. Cite NCT IDs and variant pathogenicity verbatim."
    ),
    tools=[mcp_tools],
)

## 4. Ask the agent things that require its tools

In [ ]:
print(await agent.run("What open phase II trials do we have for relapsed B-ALL?"))

In [ ]:
print(await agent.run("How would you classify the TP53 p.R175H variant, and what's the supporting evidence?"))

In [ ]:
# A composed query that requires BOTH tools in sequence.
print(await agent.run(
    "I have a relapsed B-ALL patient with a confirmed IKZF1 p.N159S variant. "
    "Annotate the variant and tell me which open trials they may be eligible for."
))

## 5. Cleanup

In [ ]:
await credential.close()